In [5]:
import threading
import time
import numpy as np
# import matplotlib.pyplot as plt
# import ccxt
# import talib
from typing import Type
from abc import ABCMeta, abstractmethod
from typing import Union
import queue
import logging

logging.basicConfig(level=logging.DEBUG, format='%(threadName)s: %(message)s)')


class AbstractStrategy(metaclass=ABCMeta):
    @abstractmethod
    def update(self, data) -> None:
        pass

    @abstractmethod
    def htf_range_filter(self, data) -> bool:
        pass

    @abstractmethod
    def ttf_range_filter(self, data) -> bool:
        pass


class Publisher(metaclass=ABCMeta):
    def __init__(self):
        self.observers = []

    def add_observer(self, observer):
        self.observers.append(observer)

    def notify_observers(self, signal):
        for observer in self.observers:
            observer.execute(signal)


In [ ]:
class CTAStrategy(AbstractStrategy, Publisher):
    def __init__(self, cfg):
        super().__init__()
        self.symbols = cfg['symbols']
        self.TTF = cfg['TTF']
        self.HTF = cfg['HTF']
        self.dfs = {} # a fixed-size dataframe for each symbol, timeFrame pair
        self.range = {}
        self.ttf_indicators = {} # ma20, ma100, etc. for each symbol stored with a dictionary of dataframes/ series
        self.htf_indicators = {}
        # test approach for waiting for the next fractal to form
        self.__counter = {}
        # multi-threading
        self.watching = {} # {(symbol, timeFrame): bool}
        self.target 
        self.events = {}

    # TODO: get_range_regression
    def get_range_regression(self, df, period):
        # get range regression
        # df: dataframe
        # period: period to calculate range regression
        # return: range regression
        df['Range'] = df['High'] - df['Low']
        df['Range'] = df['Range'].rolling(period).mean()
        df['Range'] = df['Range'].shift(1)
        df['Range'] = df['Range'].fillna(method='bfill')
        df['Range'] = df['Range'].fillna(method='ffill')
        df['Range'] = df['Range'].fillna(0)
        return df['Range']
    
    # initialize the threading.Event objects for each symbol
    def init(self):
        for symbol in self.symbols:
            self.events[symbol] = threading.Event()
            self.watching[symbol] = False

    # TODO: initialize the fractals and other indicators for each symbol for the first time
    def init_indicators(self, data):
        # initialize the fractals and other indicators for each symbol for the first time
        # data: dataframe
        df = data.copy()
        for symbol in self.symbols:
            # todo: both TTF and HTF  indicators
            self.fractals[symbol] = Fractals(df[symbol])
            self.range[symbol] = self.get_range_regression(df[symbol], 20)
            self.indicators[symbol]['ma20'] = self.compute_ma20(df[symbol])
            self.indicators[symbol]['ma100'] = self.compute_ma100(df[symbol])
            self.indicators[symbol]['ATR'] = self.compute_ATR(df[symbol])

    # update the fractals and other indicators for each symbol when new data comes
    def update_indicators(self, symbol, timeFrame, df):
        # update the fractals and other indicators for each symbol
        self.fractals[(symbol, timeFrame)].update(df)

    # TODO: streaming talib indicators
    def compute_ma20(self, df):
        # get moving average
        # df: dataframe
        # return: moving average
        # df['MA20'] = talib.SMA(df['Close'], timeperiod=20)
        # return df['MA20']
        pass

    # TODO: wait for the next fractal to form under certain conditions
    # compare(next_fractal, range.high)
    def update(self, data) -> None: # data: {symbol, timeFrame, df}
        # always update the fractals and other indicators firstly
        df = data.df.copy()
        key = (data.symbol, data.timeFrame)

        newFractal = self.fractals[key].update(df)
        self.range[key].update(df)
        self.update_indicators(data.symbol, data.timeFrame, df)


        if self.watching[key]:  # a thread is already watching
            self.__counter += 1
            if newFractal and newFractal.type == self.target[key]:
                self.events[key].set() # set the event to notify the thread to process with the new fractal
            return

        # check if price break the ttf_ma20
        if not self.watching[key]:
            if df.iloc[-1].close > self.ttf_indicators[data.symbol]['ma20'].iloc[-1] and df.iloc[-2].close < self.ttf_indicators[data.symbol]['ma20'].iloc[-2]:
                logging.debug('Price break the ttf_ma20 from the top, start watching the next bear_fractal...')
                target = 'bear'
                self.target[key] = 'bear'
                self.watching[key] = True
                self.__counter[key] += 1
                # self.flag_fractal[key] = self.fractals[key].latest.time
                # create a new thread to wait for the next fractal to form
                task = threading.Thread(
                    target=self.keep_watching, name=str(key), args=(key, target)) 
                task.start()
            elif df.iloc[-1].close < self.ttf_indicators[data.symbol]['ma20'].iloc[-1] and df.iloc[-2].close > self.ttf_indicators[data.symbol]['ma20'].iloc[-2]:
                logging.debug('Price break the ttf_ma20 from the bottom, start watching the next bull_fractal...')
                self.watching[key] = True
                self.__counter[key] += 1
                # self.flag_fractal[key] = self.fractals[key].latest.time
                # create a new thread to wait for the next fractal to form
                task = threading.Thread(
                    target=self.keep_watching, name=str(key), args=(key, target)) 
                task.start()
        

    # TODO: override htf_range_filter

    def htf_range_filter(self, data) -> bool:
        pass

    # TODO: override ttf_range_filter
    def ttf_range_filter(self, data) -> bool:
        pass

    # TODO: generate signal
    def generate_signal(self, key) -> -1 | 0 | 1:
        pass

    def keep_watching(self, key, target): # consumer @indicators@(symbol, timeFrame)
        # Wait for next fractal to form
        # TODO: correspond the TTF Range Break direction with the bear_fractal, bull_fractal
        # while len(self.fractals.latest.time) != self.flag_fractal or self.__counter < 30 or (data.iloc[-1]['close'] < self.ma100.iloc[-1]):
        #     time.sleep(1)
        cond1 = self.__counter < 30
        cond2 = self.dfs[key].iloc[-1]['close'] < self.ma100.iloc[-1]
        while self.watching[key] and cond1 and cond2:
            # wait for event to be set and consume the updated indicators
            logging.debug(f'Waiting for the next {target} fractal to form...')
            self.events[key].wait()
            logging.debug('The next fractal @{self.fractals[key].iloc[-1]} has formed, start processing...')
            # TODO: check if the next fractal is valid && generate signal
            signal = self.generate_signal(key)
            # Notify observers of new signal
            if not signal:
                self.notify_observers(signal)
            self.events[key].clear()  # Reset event? 

        # Reset waiting_for_fractal flag
        self.watching[key] = False
